In [ ]:
!pip install -q langchain langchain-community langchain-chroma langchain-huggingface pypdf pdfplumber sentence-transformers chromadb groq streamlit langchain_groq
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121  # Colab GPU 版本
!pip install -q pymupdf

print("✅ 所有套件安裝完成")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 117.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# 設定路徑變數
BASE_DIR = '/content/drive/MyDrive/rag_travel_insurance'
PDF_DIR = os.path.join(BASE_DIR, 'data/raw_pdfs')
DB_DIR = os.path.join(BASE_DIR, 'db/chroma_travel_insurance')

os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(DB_DIR, exist_ok=True)

print("Google Drive 已掛載，專案目錄確認完成。")

Mounted at /content/drive
Google Drive 已掛載，專案目錄確認完成。


In [ ]:
import os
import glob
import fitz  # PyMuPDF 套件
from langchain_core.documents import Document

def load_and_clean_pdfs_advanced(pdf_dir):
    pdf_files = glob.glob(os.path.join(pdf_dir, "*.pdf"))

    if not pdf_files:
        print(f"警告：在 {pdf_dir} 中找不到 PDF 檔案。")
        return []

    cleaned_docs = []
    for file_path in pdf_files:
        try:
            doc = fitz.open(file_path)
            full_text = ""

            for page in doc:
                # 使用 "blocks" 模式讀取，PyMuPDF 會自動分析版面區塊並排序（通常能解決雙欄問題）
                blocks = page.get_text("blocks")

                # blocks 的結構為元組，第 7 個元素 (index 6) 若為 0 代表該區塊是文字
                text_blocks = [b[4] for b in blocks if b[6] == 0]

                # 將該頁的文字區塊合併
                page_text = "\n".join(text_blocks)
                full_text += page_text + "\n\n"

            # 將單一 PDF 儲存為一個 Document，並記錄來源
            cleaned_docs.append(Document(page_content=full_text, metadata={"source": file_path}))
            doc.close()

        except Exception as e:
            print(f"讀取檔案 {file_path} 時發生錯誤: {e}")

    return cleaned_docs

# 執行新的載入函數 (請確認 PDF_DIR 變數已正確指向您的資料夾)
docs = load_and_clean_pdfs_advanced(PDF_DIR)
print(f"清洗 {len(docs)} 份 PDF 檔案。")

清洗 1 份 PDF 檔案。


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150, # 稍微增加一點重疊，確保條文跨區塊時的語意連貫
    separators=[
        r"(?=\n第[一二三四五六七八九十百]+章\s)", # 最高優先級：按「章」切
        r"(?=\n第[一二三四五六七八九十百]+條\s)", # 次高優先級：按「條」切
        "\n\n",
        "\n",
        "。",
        "；"
    ],
    is_separator_regex=True # 關鍵：告訴 LangChain 這些是正規表達式
)

chunks = text_splitter.split_documents(docs)
print(f"✅ 文件已按法規結構分割為 {len(chunks)} 個區塊")

✅ 文件已按法規結構分割為 107 個區塊


In [ ]:
import os
import shutil
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda'}
)

# 設定本地端暫存路徑與 Google Drive 最終儲存路徑
local_db_path = "/content/temp_chroma_db"
drive_db_path = "/content/drive/MyDrive/rag_travel_insurance/db/chroma_travel_insurance"

# 確保本地端為乾淨狀態
if os.path.exists(local_db_path):
    shutil.rmtree(local_db_path)

print("開始於本地端建立向量資料庫，這將避免 Google Drive 的寫入鎖定問題...")

# 在本地磁碟建立資料庫 (速度較快且穩定)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=local_db_path
)

print("本地資料庫建立完畢，正在備份至 Google Drive...")

# 確保 Google Drive 目的端乾淨，並將資料複製過去
if os.path.exists(drive_db_path):
    shutil.rmtree(drive_db_path)

shutil.copytree(local_db_path, drive_db_path)

print("向量資料庫已成功備份至 Google Drive。")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

開始於本地端建立向量資料庫，這將避免 Google Drive 的寫入鎖定問題...
本地資料庫建立完畢，正在備份至 Google Drive...
向量資料庫已成功備份至 Google Drive。


In [ ]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents.stuff import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

# 設定 Groq API Key（請確認已正確填入）
import os
os.environ["GROQ_API_KEY"] = " "   # ← 請務必替換為有效金鑰

# === 修正重點：更換為官方推薦模型 ===
llm = ChatGroq(
    model="llama-3.3-70b-versatile",   # ← 官方替代模型
    temperature=0
)

system_prompt = (
    """你是一位專業、嚴謹且具備高度洞察力的「AI 旅平險理賠專員」。
請嚴格依據以下提供的 <context> 文件內容，針對使用者的情境進行理賠評估與回答。

【回答守則】
1. 絕對客觀：必須100%基於提供的 <context> 內容回答，絕對不能憑空捏造、過度推論或使用內部知識。
2. 誠實回報：若提供的內容中完全沒有提到使用者詢問的資訊，請直接回答：「根據目前檢索到的條款資料，並未提及相關規定」，絕對不可附上無效來源。
3. 主動防坑（關鍵）：如果發現使用者的情境極有可能觸發「除外責任」或「不保事項」，必須在回答中【強烈提醒】，主動點出保單的盲區，打破使用者的錯誤期待。
4. 精準引用：只有在確實找到具體答案時，才可以在句末或段落標註具體的「條款名稱與第幾條」（例如：根據《海外突發疾病醫療健康保險附約》第二條...）。

<context>
{context}
</context>"""
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

retriever = vectorstore.as_retriever(search_kwargs={"k": 6})
qa_chain = create_retrieval_chain(
    retriever,
    create_stuff_documents_chain(llm, prompt)
)

print("✅ RAG 管線已成功更新為 llama-3.3-70b-versatile 模型")

✅ RAG 管線已成功更新為 llama-3.3-70b-versatile 模型


In [ ]:
query = "我在歐洲搭火車時，因為自己顧著講電話，結果放在旁邊座位的 10 萬塊名牌包跟蘋果筆電被小偷順手牽羊了。請問『行李損失』會全額理賠嗎？"

# 取得檢索到的區塊
retrieved_docs = retriever.invoke(query)
print("=== 系統實際檢索到的 6 個最相關區塊 ===")
for i, doc in enumerate(retrieved_docs):
    print(f"\n【文件 {i+1}】來源：{doc.metadata['source']}")
    print(doc.page_content[:500] + "..." if len(doc.page_content) > 500 else doc.page_content)

# 正式回答
result = qa_chain.invoke({"input": query})
print("\n=== 最終回答 ===")
print(result["answer"])

=== 系統實際檢索到的 6 個最相關區塊 ===

【文件 1】來源：/content/drive/MyDrive/rag_travel_insurance/data/raw_pdfs/華南產物旅行綜合保險條款.pdf
第三十三條 承保範圍 
被保險人於海外旅行期間內，其隨行託運並取得託運
行李領取單之個人行李因公共交通工具業者之處理失
當，致其在抵達目的地六小時後仍未領得時，本公司
依本契約約定之保險金額給付保險金，但保險期間內
以給付二次為限。 
 
第三十四條 特別不保事項 
對於下列事故與物品，本公司不負理賠責任： 
一、被保險人於返回中華民國境內機場之行李延誤。 
二、被保險人於返回居住所之行李延誤。 
三、被保險人事先運送之行李，或非隨身託運而分開
郵寄或運送之物品。 
 
第三十五條 理賠文件 
被保險人向本公司申請理賠時，應檢具下列文件： 
一、理賠申請書。 
二、公共交通工具業者所出具行李延誤證明。 
三、延誤達六小時以上之證明文件，必要時本公司得
要求提供行李條或托運憑證。 
 
第五節 行李損失保險(定額給付) 
第三十六條 承保範圍 
被保險人於海外旅行期間內，因下列事故致其所擁有
且置於行李箱、手提箱或類似容器內之個人物品遭受
損失，本公司依本契約約定之保險金額給付保險金，
但保險期間內以給付二次為限。 
一、竊盜、強盜與搶奪所致之毀損或滅失。 
二、交由所搭乘之公共交通工具...

【文件 2】來源：/content/drive/MyDrive/rag_travel_insurance/data/raw_pdfs/華南產物旅行綜合保險條款.pdf
第三十七條 特別不保事項（物品） 
對於下列物品之損失，本公司不負理賠責任： 
一、商業用或營業用物品、食物、動植物、機動車、
船舶、其他交通工具（包括前述交通工具之零配件）、
家具、古董、珠寶、行動電話、飾品。 
二、貨幣、股票、債券、郵票、票據、入場券、車票、
機票、船票、其他交通工具票證、有價證券及旅行文
件。 
三、文稿、圖畫、圖案、模型、樣品、帳簿或其他商
業憑證簿冊。 
四、違禁品或非法之物品。 
五、被保險人事先運送之行李，或非隨身託運而分開
郵寄或運送之物品。 

六、行李箱、手提箱或類似容器。 
七、被保險人所租用之設備。 
八、儲存或記載於磁帶、磁碟、磁片、卡片或其他供
資